# Goal 1 — recover bootstrap tables and figures from saved OOF predictions

This notebook is intentionally self-contained.

It does **not** rerun nested cross-validation or QCHAN extraction. It uses the already-saved out-of-fold prediction files produced by Goal 1 Sections 8–9, recomputes participant-cluster bootstrap confidence intervals, reloads permutation-null tables if available, and regenerates the publication figures.

Use this notebook only to recover the reporting layer after the empty-CSV state error.


In [1]:
from pathlib import Path
from collections import OrderedDict
from datetime import datetime, timezone
import json
import math

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from scipy import stats
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
)
from statsmodels.api import GLM, add_constant
from statsmodels.genmod.families import Binomial
from statsmodels.nonparametric.smoothers_lowess import lowess

FINAL_INFERENCE_MODE = False
BASE_SEED = 20260825
OUTER_REPEATS = 10
N_BOOTSTRAPS = 2000 if FINAL_INFERENCE_MODE else 500

def find_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "outputs" / "goal1").exists():
            return candidate
    raise FileNotFoundError("Could not locate Paper 2 repository root.")

ROOT = find_root()
RUN_TAG = "final" if FINAL_INFERENCE_MODE else "development"
OUT = ROOT / "outputs" / "goal1" / RUN_TAG
OOF_DIR = OUT / "oof"
TABLES = OUT / "tables"
FIGURES = OUT / "figures"

for d in [OOF_DIR, TABLES, FIGURES]:
    d.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("RUN:", RUN_TAG)
print("OOF:", OOF_DIR)


ROOT: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code
RUN: development
OOF: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\outputs\goal1\development\oof


## 1. Inventory saved OOF files

In [2]:
oof_files = sorted(OOF_DIR.glob("*.csv"))
inventory = pd.DataFrame([
    {"file": p.name, "bytes": p.stat().st_size}
    for p in oof_files
])
display(inventory)

assert len(oof_files) > 0, "No saved OOF files found."
assert (inventory["bytes"] > 0).all(), "At least one OOF file is empty."


""


AssertionError: No saved OOF files found.

## 2. Load diagnosis and severity OOF predictions

In [3]:
MODEL_NAMES = [
    "Age",
    "Support-only",
    "QADD",
    "QGAIN",
    "QREV",
    "QCHAN",
    "Core-Q",
    "Age + Core-Q",
    "Core-Q HGB",
]

def safe_name(name):
    return name.lower().replace(" ", "_").replace("+", "plus")

def load_task(task):
    results = OrderedDict()
    problems = []

    for model in MODEL_NAMES:
        path = OOF_DIR / f"{task}_{safe_name(model)}.csv"
        if not path.exists():
            # HGB may have been intentionally skipped.
            if model == "Core-Q HGB":
                continue
            problems.append((model, "missing", str(path)))
            continue

        if path.stat().st_size == 0:
            problems.append((model, "empty", str(path)))
            continue

        df = pd.read_csv(path)
        required = {
            "model", "task", "repeat", "outer_fold",
            "participant_id", "logical_recording_id", "y", "prediction",
        }
        if not required.issubset(df.columns):
            problems.append((model, "bad schema", df.columns.tolist()))
            continue

        df["participant_id"] = df["participant_id"].astype(str)
        assert df.groupby(["participant_id", "repeat"]).size().eq(1).all()
        assert df["repeat"].nunique() == 10

        results[model] = df

    if problems:
        display(pd.DataFrame(problems, columns=["model", "problem", "detail"]))

    return results

diagnosis_results = load_task("diagnosis")
severity_results = load_task("severity")

print("Diagnosis models loaded:", list(diagnosis_results))
print("Severity models loaded:", list(severity_results))

assert "Core-Q" in diagnosis_results
assert "Core-Q" in severity_results
assert "Age" in diagnosis_results and "Age + Core-Q" in diagnosis_results
assert "Age" in severity_results and "Age + Core-Q" in severity_results


,model,problem,detail
0,Age,missing,C:\Users\musikicn\Desktop\Nevena_project\Paper...
1,Support-only,missing,C:\Users\musikicn\Desktop\Nevena_project\Paper...
2,QADD,missing,C:\Users\musikicn\Desktop\Nevena_project\Paper...
3,QGAIN,missing,C:\Users\musikicn\Desktop\Nevena_project\Paper...
4,QREV,missing,C:\Users\musikicn\Desktop\Nevena_project\Paper...
5,QCHAN,missing,C:\Users\musikicn\Desktop\Nevena_project\Paper...
6,Core-Q,missing,C:\Users\musikicn\Desktop\Nevena_project\Paper...
7,Age + Core-Q,missing,C:\Users\musikicn\Desktop\Nevena_project\Paper...


,model,problem,detail
0,Age,missing,C:\Users\musikicn\Desktop\Nevena_project\Paper...
1,Support-only,missing,C:\Users\musikicn\Desktop\Nevena_project\Paper...
2,QADD,missing,C:\Users\musikicn\Desktop\Nevena_project\Paper...
3,QGAIN,missing,C:\Users\musikicn\Desktop\Nevena_project\Paper...
4,QREV,missing,C:\Users\musikicn\Desktop\Nevena_project\Paper...
5,QCHAN,missing,C:\Users\musikicn\Desktop\Nevena_project\Paper...
6,Core-Q,missing,C:\Users\musikicn\Desktop\Nevena_project\Paper...
7,Age + Core-Q,missing,C:\Users\musikicn\Desktop\Nevena_project\Paper...


Diagnosis models loaded: []
Severity models loaded: []


AssertionError: 

## 3. Recompute observed metrics from OOF predictions

In [4]:
def repeat_metrics(oof, task):
    rows = []
    for repeat, g in oof.groupby("repeat"):
        y = g["y"].to_numpy(float)
        p = g["prediction"].to_numpy(float)

        if task == "diagnosis":
            rows.append({
                "repeat": repeat,
                "AUROC": roc_auc_score(y, p),
                "Brier": brier_score_loss(y, p),
                "AUPRC": average_precision_score(y, p),
            })
        else:
            rows.append({
                "repeat": repeat,
                "MAE": mean_absolute_error(y, p),
                "RMSE": math.sqrt(mean_squared_error(y, p)),
                "Spearman_rho": stats.spearmanr(y, p).statistic,
                "R2": r2_score(y, p),
            })
    return pd.DataFrame(rows)

def mean_metrics(oof, task):
    m = repeat_metrics(oof, task)
    return {c: float(m[c].mean()) for c in m.columns if c != "repeat"}

diagnosis_observed = pd.DataFrame([
    {"model": name, "n_participants": oof["participant_id"].nunique(),
     **mean_metrics(oof, "diagnosis")}
    for name, oof in diagnosis_results.items()
])

severity_observed = pd.DataFrame([
    {"model": name, "n_participants": oof["participant_id"].nunique(),
     **mean_metrics(oof, "severity")}
    for name, oof in severity_results.items()
])

diagnosis_observed.to_csv(TABLES / "diagnosis_observed_metrics.csv", index=False)
severity_observed.to_csv(TABLES / "severity_observed_metrics.csv", index=False)

display(diagnosis_observed)
display(severity_observed)


""


""


## 4. Recompute participant-cluster bootstrap CIs

In [5]:
def metric_value(task, metric, y, pred):
    if task == "diagnosis":
        if metric == "AUROC": return roc_auc_score(y, pred)
        if metric == "Brier": return brier_score_loss(y, pred)
        if metric == "AUPRC": return average_precision_score(y, pred)
    else:
        if metric == "MAE": return mean_absolute_error(y, pred)
        if metric == "RMSE": return math.sqrt(mean_squared_error(y, pred))
        if metric == "Spearman_rho": return stats.spearmanr(y, pred).statistic
        if metric == "R2": return r2_score(y, pred)
    raise ValueError((task, metric))

def bootstrap_distribution(oof, task, metric, B, seed):
    participants = np.array(sorted(oof["participant_id"].unique()))
    repeat_tables = {r: g.set_index("participant_id") for r, g in oof.groupby("repeat")}
    rng = np.random.default_rng(seed)
    values = []

    while len(values) < B:
        sampled = rng.choice(participants, size=len(participants), replace=True)
        per_repeat = []
        valid = True

        for _, g in repeat_tables.items():
            boot = g.loc[sampled]
            y = boot["y"].to_numpy(float)
            p = boot["prediction"].to_numpy(float)

            # A rare diagnosis bootstrap can contain one class only.
            if task == "diagnosis" and metric in {"AUROC", "AUPRC"} and len(np.unique(y)) < 2:
                valid = False
                break

            per_repeat.append(metric_value(task, metric, y, p))

        if valid and np.all(np.isfinite(per_repeat)):
            values.append(float(np.mean(per_repeat)))

    return np.asarray(values)

def ci(values):
    return float(np.quantile(values, .025)), float(np.quantile(values, .975))

diag_rows = []
for i, (name, oof) in enumerate(diagnosis_results.items()):
    for metric in ["AUROC", "Brier", "AUPRC"]:
        dist = bootstrap_distribution(
            oof, "diagnosis", metric, N_BOOTSTRAPS,
            BASE_SEED + 1000 + i * 10
        )
        lo, hi = ci(dist)
        diag_rows.append({
            "model": name,
            "metric": metric,
            "estimate": mean_metrics(oof, "diagnosis")[metric],
            "ci_low": lo,
            "ci_high": hi,
            "bootstrap_replicates": N_BOOTSTRAPS,
        })

sev_rows = []
for i, (name, oof) in enumerate(severity_results.items()):
    for metric in ["MAE", "RMSE", "Spearman_rho", "R2"]:
        dist = bootstrap_distribution(
            oof, "severity", metric, N_BOOTSTRAPS,
            BASE_SEED + 5000 + i * 10
        )
        lo, hi = ci(dist)
        sev_rows.append({
            "model": name,
            "metric": metric,
            "estimate": mean_metrics(oof, "severity")[metric],
            "ci_low": lo,
            "ci_high": hi,
            "bootstrap_replicates": N_BOOTSTRAPS,
        })

diagnosis_ci = pd.DataFrame(diag_rows)
severity_ci = pd.DataFrame(sev_rows)

assert len(diagnosis_ci) > 0
assert len(severity_ci) > 0
assert "metric" in diagnosis_ci.columns
assert "metric" in severity_ci.columns

diagnosis_ci.to_csv(TABLES / "diagnosis_bootstrap_ci.csv", index=False)
severity_ci.to_csv(TABLES / "severity_bootstrap_ci.csv", index=False)

print("BOOTSTRAP RECOVERY: PASS")
print("diagnosis_bootstrap_ci.csv bytes:",
      (TABLES / "diagnosis_bootstrap_ci.csv").stat().st_size)
print("severity_bootstrap_ci.csv bytes:",
      (TABLES / "severity_bootstrap_ci.csv").stat().st_size)

display(diagnosis_ci.loc[diagnosis_ci["metric"].eq("AUROC")])
display(severity_ci.loc[severity_ci["metric"].eq("MAE")])


AssertionError: 

## 5. Paired Age + Core-Q versus Age bootstrap contrast

In [6]:
def paired_bootstrap(oof_a, oof_b, task, metric, B, seed):
    common = sorted(set(oof_a["participant_id"]) & set(oof_b["participant_id"]))
    merged = oof_a.loc[oof_a["participant_id"].isin(common),
                       ["participant_id", "repeat", "y", "prediction"]].merge(
        oof_b.loc[oof_b["participant_id"].isin(common),
                  ["participant_id", "repeat", "y", "prediction"]],
        on=["participant_id", "repeat"],
        validate="one_to_one",
        suffixes=("_a", "_b")
    )
    rng = np.random.default_rng(seed)
    participants = np.array(common)
    values = []

    while len(values) < B:
        sampled = rng.choice(participants, len(participants), replace=True)
        diffs = []
        valid = True

        for repeat in range(1, 11):
            g = merged.loc[merged["repeat"].eq(repeat)].set_index("participant_id").loc[sampled]
            y = g["y_a"].to_numpy(float)

            if task == "diagnosis" and metric == "AUROC" and len(np.unique(y)) < 2:
                valid = False
                break

            a = metric_value(task, metric, y, g["prediction_a"].to_numpy(float))
            b = metric_value(task, metric, y, g["prediction_b"].to_numpy(float))
            diffs.append(b - a)

        if valid:
            values.append(float(np.mean(diffs)))

    return np.asarray(values)

diag_delta = paired_bootstrap(
    diagnosis_results["Age"], diagnosis_results["Age + Core-Q"],
    "diagnosis", "AUROC", N_BOOTSTRAPS, BASE_SEED + 8001
)
sev_delta = paired_bootstrap(
    severity_results["Age"], severity_results["Age + Core-Q"],
    "severity", "MAE", N_BOOTSTRAPS, BASE_SEED + 8002
)

dlo, dhi = ci(diag_delta)
slo, shi = ci(sev_delta)

paired = pd.DataFrame([
    {
        "task": "diagnosis",
        "contrast": "Age + Core-Q minus Age",
        "metric": "AUROC",
        "estimate": mean_metrics(diagnosis_results["Age + Core-Q"], "diagnosis")["AUROC"]
                  - mean_metrics(diagnosis_results["Age"], "diagnosis")["AUROC"],
        "ci_low": dlo,
        "ci_high": dhi,
        "n_participants": diagnosis_results["Age"]["participant_id"].nunique(),
    },
    {
        "task": "severity",
        "contrast": "Age + Core-Q minus Age",
        "metric": "MAE",
        "estimate": mean_metrics(severity_results["Age + Core-Q"], "severity")["MAE"]
                  - mean_metrics(severity_results["Age"], "severity")["MAE"],
        "ci_low": slo,
        "ci_high": shi,
        "n_participants": severity_results["Age"]["participant_id"].nunique(),
    },
])
paired.to_csv(TABLES / "age_incremental_paired_contrast.csv", index=False)
display(paired)


KeyError: 'Age'

## 6. Reload permutation nulls

In [7]:
perm_paths = {
    "diagnosis": TABLES / "diagnosis_coreq_permutation_null.csv",
    "severity": TABLES / "severity_coreq_permutation_null.csv",
}
permutation_tables = {}

for task, path in perm_paths.items():
    if path.exists() and path.stat().st_size > 0:
        df = pd.read_csv(path)
        required = {"permutation", "null_metric", "observed_metric", "empirical_p"}
        assert required.issubset(df.columns), (task, df.columns.tolist())
        permutation_tables[task] = df
        print(task, "permutations:", len(df), "| empirical P:", df["empirical_p"].iloc[0])
    else:
        print(task, "permutation table missing/empty:", path)

if set(permutation_tables) != {"diagnosis", "severity"}:
    display(Markdown(
        "**Permutation tables are not both available.** "
        "Bootstrap recovery is complete, but Main Figure 2 panels b/d "
        "must wait until Goal 1 Section 11 is rerun."
    ))


diagnosis permutation table missing/empty: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\outputs\goal1\development\tables\diagnosis_coreq_permutation_null.csv
severity permutation table missing/empty: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\outputs\goal1\development\tables\severity_coreq_permutation_null.csv


**Permutation tables are not both available.** Bootstrap recovery is complete, but Main Figure 2 panels b/d must wait until Goal 1 Section 11 is rerun.

## 7. Publication figure styling

In [8]:
MM_TO_IN = 1 / 25.4
DOUBLE_COLUMN_IN = 183 * MM_TO_IN
SINGLE_COLUMN_IN = 89 * MM_TO_IN

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.labelsize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "legend.fontsize": 6,
    "axes.linewidth": .7,
    "xtick.major.width": .6,
    "ytick.major.width": .6,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

def clean_axis(ax):
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(direction="out")

def panel_label(ax, label):
    ax.text(-.12, 1.04, label, transform=ax.transAxes,
            fontsize=8, fontweight="bold", va="bottom")

def save_figure(fig, stem, source_tables):
    fig.savefig(FIGURES / f"{stem}.pdf", bbox_inches="tight")
    fig.savefig(FIGURES / f"{stem}.svg", bbox_inches="tight")
    fig.savefig(FIGURES / f"{stem}.png", dpi=600, bbox_inches="tight")
    plt.close(fig)
    (FIGURES / f"{stem}.provenance.json").write_text(
        json.dumps({
            "stem": stem,
            "created_utc": datetime.now(timezone.utc).isoformat(),
            "run_tag": RUN_TAG,
            "source_tables": source_tables,
        }, indent=2),
        encoding="utf-8",
    )


## 8. Main Figure 2

In [9]:
if set(permutation_tables) != {"diagnosis", "severity"}:
    raise RuntimeError(
        "Permutation tables are missing or empty. "
        "Rerun Goal 1 Section 11 before generating the full main figure."
    )

order = [m for m in MODEL_NAMES if m in diagnosis_results]

diag_plot = (
    diagnosis_ci.loc[diagnosis_ci["metric"].eq("AUROC")]
    .set_index("model").loc[order].reset_index()
)
sev_plot = (
    severity_ci.loc[severity_ci["metric"].eq("MAE")]
    .set_index("model").loc[order].reset_index()
)

diag_plot["n_participants"] = diag_plot["model"].map({
    m: diagnosis_results[m]["participant_id"].nunique() for m in order
})
sev_plot["n_participants"] = sev_plot["model"].map({
    m: severity_results[m]["participant_id"].nunique() for m in order
})

diag_plot.to_csv(TABLES / "figure2a_diagnosis_ladder_source.csv", index=False)
sev_plot.to_csv(TABLES / "figure2c_severity_ladder_source.csv", index=False)

fig, axes = plt.subplots(
    2, 2,
    figsize=(DOUBLE_COLUMN_IN, 158 * MM_TO_IN),
    constrained_layout=True
)

ax = axes[0, 0]
y = np.arange(len(diag_plot))
x = diag_plot["estimate"].to_numpy(float)
xe = np.vstack([
    x - diag_plot["ci_low"].to_numpy(float),
    diag_plot["ci_high"].to_numpy(float) - x,
])
ax.errorbar(x, y, xerr=xe, fmt="o", capsize=2, linewidth=.8, markersize=3.5)
ax.axvline(.5, linestyle="--", linewidth=.7)
ax.set_yticks(y)
ax.set_yticklabels([f"{m} (n={n})" for m, n in zip(diag_plot["model"], diag_plot["n_participants"])])
ax.invert_yaxis()
ax.set_xlabel("AUROC")
ax.set_xlim(.35, 1)
clean_axis(ax); panel_label(ax, "a")

ax = axes[0, 1]
nd = permutation_tables["diagnosis"]
ax.hist(nd["null_metric"], bins=min(30, max(8, len(nd)//3)), density=True)
obs = float(nd["observed_metric"].iloc[0])
ax.axvline(obs, linewidth=1)
ax.axvline(.5, linestyle="--", linewidth=.7)
ax.set_xlabel("Core-Q AUROC under participant-level null")
ax.set_ylabel("Density")
ax.text(.98, .96,
        f"Observed AUROC = {obs:.3f}\nEmpirical P = {float(nd['empirical_p'].iloc[0]):.4f}\nB = {len(nd)}",
        transform=ax.transAxes, ha="right", va="top")
clean_axis(ax); panel_label(ax, "b")

ax = axes[1, 0]
y = np.arange(len(sev_plot))
x = sev_plot["estimate"].to_numpy(float)
xe = np.vstack([
    x - sev_plot["ci_low"].to_numpy(float),
    sev_plot["ci_high"].to_numpy(float) - x,
])
ax.errorbar(x, y, xerr=xe, fmt="o", capsize=2, linewidth=.8, markersize=3.5)
ax.set_yticks(y)
ax.set_yticklabels([f"{m} (n={n})" for m, n in zip(sev_plot["model"], sev_plot["n_participants"])])
ax.invert_yaxis()
ax.set_xlabel("MAE (ALSFRS-R bulbar points)")
clean_axis(ax); panel_label(ax, "c")

ax = axes[1, 1]
ns = permutation_tables["severity"]
ax.hist(ns["null_metric"], bins=min(30, max(8, len(ns)//3)), density=True)
obs = float(ns["observed_metric"].iloc[0])
ax.axvline(obs, linewidth=1)
ax.set_xlabel("Core-Q MAE under participant-level null (points)")
ax.set_ylabel("Density")
ax.text(.98, .96,
        f"Observed MAE = {obs:.3f}\nEmpirical P = {float(ns['empirical_p'].iloc[0]):.4f}\nB = {len(ns)}",
        transform=ax.transAxes, ha="right", va="top")
clean_axis(ax); panel_label(ax, "d")

stem = f"Figure2_Goal1_information_availability_{RUN_TAG}"
save_figure(fig, stem, [
    "figure2a_diagnosis_ladder_source.csv",
    "diagnosis_coreq_permutation_null.csv",
    "figure2c_severity_ladder_source.csv",
    "severity_coreq_permutation_null.csv",
])

print("FIGURE 2: PASS")
print(FIGURES / f"{stem}.pdf")


RuntimeError: Permutation tables are missing or empty. Rerun Goal 1 Section 11 before generating the full main figure.